# Differential Equations — Session 40
## Section 9.1: Euler Methods and Error Analysis

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to derive Euler's method from a tangent-line approximation, distinguish round-off and truncation errors, define local and global error, use big-$O$ notation, state Euler and improved-Euler convergence orders, estimate observed order numerically, and recognize step-size restrictions caused by stability.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Euler's geometric derivation |
| 18–35 min | Local and global truncation errors |
| 35–50 min | Big-$O$ notation and convergence order |
| 50–68 min | Improved Euler predictor–corrector |
| 68–84 min | Error experiments and stability |
| 84–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=8, suppress=True)
def euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n): ys[k+1]=ys[k]+h*f(xs[k],ys[k])
    return xs,ys
def improved_euler(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        p=ys[k]+h*f(xs[k],ys[k])
        ys[k+1]=ys[k]+0.5*h*(f(xs[k],ys[k])+f(xs[k+1],p))
    return xs,ys
def midpoint_rk2(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        ys[k+1]=ys[k]+h*k2
    return xs,ys
def rk4(f, x0, y0, h, n):
    xs=x0+h*np.arange(n+1); ys=np.zeros(n+1); ys[0]=y0
    for k in range(n):
        k1=f(xs[k],ys[k]); k2=f(xs[k]+h/2,ys[k]+h*k1/2)
        k3=f(xs[k]+h/2,ys[k]+h*k2/2); k4=f(xs[k]+h,ys[k]+h*k3)
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return xs,ys
def rk4_system(f,t0,y0,h,n):
    ts=t0+h*np.arange(n+1); y0=np.asarray(y0,float)
    ys=np.zeros((n+1,len(y0))); ys[0]=y0
    for k in range(n):
        k1=np.asarray(f(ts[k],ys[k]))
        k2=np.asarray(f(ts[k]+h/2,ys[k]+h*k1/2))
        k3=np.asarray(f(ts[k]+h/2,ys[k]+h*k2/2))
        k4=np.asarray(f(ts[k]+h,ys[k]+h*k3))
        ys[k+1]=ys[k]+h*(k1+2*k2+2*k3+k4)/6
    return ts,ys
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 9.1-A — Euler's method

For

$$
y'=f(x,y),\qquad y(x_0)=y_0,
$$

with $x_n=x_0+nh$,

$$
y_{n+1}=y_n+h f(x_n,y_n).
$$

### Definition 9.1-B — Round-off error

Round-off error is caused by representing and operating on numbers with finitely many digits.

### Definition 9.1-C — Local truncation error

The local truncation error is the one-step error obtained when the exact value $y(x_n)$ is used at the beginning of the step.

### Definition 9.1-D — Global error

$$
E_n=y(x_n)-y_n.
$$

It includes accumulated truncation and round-off errors.

### Definition 9.1-E — Order notation

An error $e(h)$ is $O(h^p)$ when constants $C,h_0>0$ exist such that

$$
|e(h)|\le Ch^p
$$

for $0<h<h_0$.

### Theorem 9.1-F — Euler error order

Under standard smoothness and Lipschitz assumptions:

- local truncation error is $O(h^2)$;
- global truncation error is $O(h)$.

### Definition 9.1-G — Improved Euler method

The predictor is

$$
y_{n+1}^{(p)}=y_n+h f(x_n,y_n).
$$

The corrected value is

$$
y_{n+1}
=
y_n+\frac h2
\left[
f(x_n,y_n)+f(x_{n+1},y_{n+1}^{(p)})
\right].
$$

### Theorem 9.1-H — Improved Euler order

Its local truncation error is $O(h^3)$ and global error is $O(h^2)$.

### Classroom Checkpoint — Error Orders

State the local and global truncation-error orders of Euler’s method.

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Tangent-line geometry

Euler's method replaces the solution curve over one step by its tangent line at the left endpoint.

In [ ]:
f = lambda x, y: y
x0, y0, h = 0.0, 1.0, 0.8
x = np.linspace(0, h, 300)
exact = np.exp(x)
tangent = y0 + f(x0,y0)*(x-x0)

plt.plot(x, exact, label=r"exact $e^x$")
plt.plot(x, tangent, linestyle="--", label="Euler tangent")
plt.scatter([h], [y0+h*f(x0,y0)], s=80, label="Euler endpoint")
plt.legend()
plt.title("One Euler step")
plt.show()

In [ ]:
def euler_explorer(h=0.5):
    f = lambda x, y: y
    final = 3.0
    n = int(round(final/h))
    h = final/n
    xs, ys = euler(f, 0, 1, h, n)
    grid = np.linspace(0, final, 700)

    plt.plot(grid, np.exp(grid), label="exact")
    plt.plot(xs, ys, marker="o", label=fr"Euler, $h={h:.3f}$")
    plt.legend()
    plt.title("Euler polygonal approximation")
    plt.show()
    print("final error:", abs(np.exp(final)-ys[-1]))

if WIDGETS_AVAILABLE:
    interact(euler_explorer, h=FloatSlider(min=0.05,max=1.0,step=0.05,value=0.5))
else:
    euler_explorer()

## 2. Taylor derivation of the local error

Taylor's theorem gives

$$
y(x_{n+1})
=
y(x_n)+hy'(x_n)+\frac{h^2}{2}y''(\xi_n).
$$

Euler discards the remainder, so the local defect is

$$
\tau_{n+1}
=
\frac{h^2}{2}y''(\xi_n).
$$

In [ ]:
h_values = np.array([0.8,0.4,0.2,0.1,0.05,0.025])
local_errors = np.abs(np.exp(h_values)-(1+h_values))

plt.loglog(h_values, local_errors, marker="o", label="measured local error")
plt.loglog(h_values, 0.5*h_values**2, linestyle="--", label=r"$0.5h^2$")
plt.xlabel("h")
plt.ylabel("one-step error")
plt.legend()
plt.title("Euler local error is second order")
plt.show()

## 3. Global order experiment

For $y'=y$, $y(0)=1$, compute the error at $x=1$ while halving $h$.

In [ ]:
hs = np.array([1/5,1/10,1/20,1/40,1/80,1/160])
errors = []
for h in hs:
    n = int(round(1/h))
    _, ys = euler(lambda x,y:y, 0, 1, h, n)
    errors.append(abs(np.e-ys[-1]))
errors = np.array(errors)

observed = np.log(errors[:-1]/errors[1:])/np.log(2)
print("h values:", hs)
print("errors:", errors)
print("observed orders:", observed)

plt.loglog(hs, errors, marker="o", label="Euler global error")
plt.loglog(hs, errors[-1]*(hs/hs[-1]), linestyle="--", label=r"reference $O(h)$")
plt.legend()
plt.show()

## 4. Improved Euler averages two slopes

In [ ]:
def compare_euler_methods(h=0.25):
    f = lambda x,y: 2*x*y
    exact = lambda x: np.exp(x**2-1)
    x0, final, y0 = 1.0, 1.5, 1.0
    n = int(round((final-x0)/h))
    h = (final-x0)/n

    xe, ye = euler(f,x0,y0,h,n)
    xi, yi = improved_euler(f,x0,y0,h,n)
    grid = np.linspace(x0,final,600)

    plt.plot(grid, exact(grid), label="exact")
    plt.plot(xe, ye, marker="o", label="Euler")
    plt.plot(xi, yi, marker="s", label="improved Euler")
    plt.legend()
    plt.title(fr"Step size $h={h:.4f}$")
    plt.show()

    print("Euler final error:", abs(exact(final)-ye[-1]))
    print("Improved Euler final error:", abs(exact(final)-yi[-1]))

if WIDGETS_AVAILABLE:
    interact(compare_euler_methods,
             h=FloatSlider(min=0.025,max=0.25,step=0.025,value=0.25))
else:
    compare_euler_methods()

## 5. Measured convergence orders

In [ ]:
f = lambda x,y: 2*x*y
exact = lambda x: np.exp(x**2-1)
hs = np.array([0.1,0.05,0.025,0.0125])
e1, e2 = [], []

for h in hs:
    n = int(round(0.5/h))
    e1.append(abs(exact(1.5)-euler(f,1,1,h,n)[1][-1]))
    e2.append(abs(exact(1.5)-improved_euler(f,1,1,h,n)[1][-1]))

e1, e2 = np.array(e1), np.array(e2)
print("Euler observed order:", np.log(e1[:-1]/e1[1:])/np.log(2))
print("Improved Euler observed order:", np.log(e2[:-1]/e2[1:])/np.log(2))

plt.loglog(hs,e1,marker="o",label="Euler")
plt.loglog(hs,e2,marker="s",label="improved Euler")
plt.legend()
plt.xlabel("h")
plt.ylabel("final error")
plt.show()

## 6. Stability on the test equation

For

$$
y'=\lambda y,
$$

Euler gives

$$
y_{n+1}=(1+h\lambda)y_n.
$$

When $\lambda<0$, decay is reproduced only if

$$
|1+h\lambda|<1.
$$

In [ ]:
def euler_stability(lambda_value=-5.0, h=0.3):
    n = 30
    xs, ys = euler(lambda x,y:lambda_value*y,0,1,h,n)
    exact = np.exp(lambda_value*xs)

    plt.plot(xs,exact,label="exact")
    plt.plot(xs,ys,marker="o",label="Euler")
    plt.legend()
    plt.title(fr"$1+h\lambda={1+h*lambda_value:.3f}$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        euler_stability,
        lambda_value=FloatSlider(min=-20,max=-0.5,step=0.5,value=-5),
        h=FloatSlider(min=0.01,max=1.0,step=0.01,value=0.3)
    )
else:
    euler_stability()

## 7. Round-off versus truncation

Reducing $h$ decreases truncation error, but it increases the number of operations. In finite precision, an excessively small step may eventually stop improving the answer.

In [ ]:
# Artificial low-precision Euler computation to make round-off visible.
def euler_low_precision(h):
    n = int(round(1/h))
    y = np.float32(1.0)
    h32 = np.float32(1/n)
    for _ in range(n):
        y = np.float32(y + h32*y)
    return float(y)

hs = 2.0**(-np.arange(2,20))
errs64, errs32 = [], []
for h in hs:
    n = int(round(1/h))
    errs64.append(abs(np.e-euler(lambda x,y:y,0,1,1/n,n)[1][-1]))
    errs32.append(abs(np.e-euler_low_precision(h)))

plt.loglog(hs,errs64,label="float64")
plt.loglog(hs,errs32,label="float32")
plt.xlabel("h")
plt.ylabel("final error")
plt.legend()
plt.title("Truncation and finite-precision effects")
plt.show()

## Classroom Checkpoint — Exit Check

Euler's global error is approximately $0.04$ when $h=0.1$. What error is expected when $h=0.05$?

> Pause here. Let students commit to an answer before running the next cell.